# Project Overview & Dataset Description
This notebook documents the cleaning and transformation process for the NYC Motor Vehicle Collisions dataset.

# Environment Setup
We begin by importing the necessary Python libraries for data manipulation and ensuring the environment is ready for large-scale data processing.

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Motor_Vehicle_Collisions_-_Crashes_2020jan1-2021dec31.csv to Motor_Vehicle_Collisions_-_Crashes_2020jan1-2021dec31.csv


In [6]:
import pandas as pd
import io

# Load your dataset
# Ensure you cite the source as NYC Open Data in your final report
df = pd.read_csv(io.BytesIO(uploaded['Motor_Vehicle_Collisions_-_Crashes_2020jan1-2021dec31.csv']), low_memory=False)

# Quick look at the structure
print(f"Initial row count: {df.shape[0]}")
df.head()

Initial row count: 223474


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,01/01/2020,13:39,BROOKLYN,11204.0,NaN,NaN,NaN,bay ridge pkwy,18 avenue,NaN,...,NaN,NaN,NaN,NaN,4267649,Box Truck,NaN,NaN,NaN,NaN
1,01/01/2020,4:20,NaN,NaN,40.762196,-73.870415,"(40.762196, -73.870415)",99 STREET,NaN,NaN,...,Other Vehicular,Other Vehicular,Other Vehicular,NaN,4269445,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,Sedan,Station Wagon/Sport Utility Vehicle,NaN
2,01/01/2020,15:41,NaN,NaN,40.780754,-73.952580,"(40.780754, -73.95258)",3 AVENUE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4267699,Station Wagon/Sport Utility Vehicle,Taxi,NaN,NaN,NaN
3,01/01/2020,19:00,QUEENS,11367.0,40.725353,-73.821080,"(40.725353, -73.82108)",MAIN STREET,72 DRIVE,NaN,...,NaN,NaN,NaN,NaN,4268937,Sedan,NaN,NaN,NaN,NaN
4,01/01/2020,14:40,QUEENS,11372.0,40.753826,-73.884120,"(40.753826, -73.88412)",34 AVENUE,83 STREET,NaN,...,Driver Inattention/Distraction,NaN,NaN,NaN,4270271,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 223474 entries, 0 to 223473
Data columns (total 29 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   CRASH DATE                     223474 non-null  str    
 1   CRASH TIME                     223474 non-null  str    
 2   BOROUGH                        145911 non-null  str    
 3   ZIP CODE                       145889 non-null  float64
 4   LATITUDE                       205769 non-null  float64
 5   LONGITUDE                      205769 non-null  float64
 6   LOCATION                       205769 non-null  str    
 7   ON STREET NAME                 164219 non-null  str    
 8   CROSS STREET NAME              104336 non-null  str    
 9   OFF STREET NAME                59254 non-null   str    
 10  NUMBER OF PERSONS INJURED      223473 non-null  float64
 11  NUMBER OF PERSONS KILLED       223474 non-null  int64  
 12  NUMBER OF PEDESTRIANS INJURED  223474 non

## TEMPORAL FILTERING
Converting string dates to datetime objects

In [7]:
df['CRASH DATE'] = pd.to_datetime(df['CRASH DATE'])
# Filtering for the 2020-2021 window requested for the analytical story
df_cleaned = df[(df['CRASH DATE'] >= '2020-01-01') & (df['CRASH DATE'] <= '2021-12-31')].copy()

## HANDLING MISSING GEOGRAPHIC DATA
Since Dashboard 2 is a 'Hotspot Map', rows without coordinates cannot be plotted.
We also drop rows missing 'BOROUGH' to ensure our Borough bar charts are accurate.

In [8]:
df_cleaned = df_cleaned.dropna(subset=['LATITUDE', 'LONGITUDE', 'BOROUGH'])

## DATA TYPE STANDARDIZATION AND COLUMN OPTIMIZATION
Fill the single missing value in 'NUMBER OF PERSONS INJURED' with 0 to prevent calculation errors
df_cleaned['NUMBER OF PERSONS INJURED'] = df_cleaned['NUMBER OF PERSONS INJURED'].fillna(0).astype(int)

Keeping relevant columns to reduce file size while meeting the >8 columns requirement

In [9]:
df_cleaned['NUMBER OF PERSONS INJURED'] = df_cleaned['NUMBER OF PERSONS INJURED'].fillna(0).astype(int)

In [10]:
usable_columns = [
    'CRASH DATE', 'CRASH TIME', 'BOROUGH', 'LATITUDE', 'LONGITUDE',
    'CONTRIBUTING FACTOR VEHICLE 1', 'NUMBER OF PERSONS INJURED',
    'NUMBER OF PERSONS KILLED', 'ON STREET NAME', 'VEHICLE TYPE CODE 1'
]

In [11]:
df_final = df_cleaned[usable_columns]
print(df_final.shape)
df_final.head()

(141727, 10)


,CRASH DATE,CRASH TIME,BOROUGH,LATITUDE,LONGITUDE,CONTRIBUTING FACTOR VEHICLE 1,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,ON STREET NAME,VEHICLE TYPE CODE 1
3,2020-01-01,19:00,QUEENS,40.725353,-73.821080,Unspecified,0,0,MAIN STREET,Sedan
4,2020-01-01,14:40,QUEENS,40.753826,-73.884120,Driver Inattention/Distraction,0,0,34 AVENUE,Station Wagon/Sport Utility Vehicle
5,2020-01-01,5:00,QUEENS,40.747097,-73.886665,Unspecified,0,0,NaN,Sedan
8,2020-01-01,17:38,BRONX,40.836212,-73.863290,Unspecified,1,0,NaN,Sedan
10,2020-01-01,0:39,BROOKLYN,40.649530,-73.916330,Traffic Control Disregarded,0,0,AVENUE B,Sedan


In [12]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 141727 entries, 3 to 223472
Data columns (total 10 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   CRASH DATE                     141727 non-null  datetime64[ns]
 1   CRASH TIME                     141727 non-null  object        
 2   BOROUGH                        141727 non-null  object        
 3   LATITUDE                       141727 non-null  float64       
 4   LONGITUDE                      141727 non-null  float64       
 5   CONTRIBUTING FACTOR VEHICLE 1  140953 non-null  object        
 6   NUMBER OF PERSONS INJURED      141727 non-null  int64         
 7   NUMBER OF PERSONS KILLED       141727 non-null  int64         
 8   ON STREET NAME                 85883 non-null   object        
 9   VEHICLE TYPE CODE 1            139985 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(5)
memory usage: 11.9+

## Final Export

In [13]:
df_final.to_csv('NYC_Collisions_Cleaned.csv', index=False)
print(f"Cleaned dataset contains {df_final.shape[0]} rows and {df_final.shape[1]} columns.")

Cleaned dataset contains 141727 rows and 10 columns.
